# 01 — Graph RAG

**Goal**: Combine knowledge graphs with LLMs for better reasoning over connected information.

Graph RAG = RAG over a knowledge graph instead of a flat vector store.

In [ ]:
# !pip install networkx

import networkx as nx
import requests
import numpy as np

def ollama_generate(prompt):
    resp = requests.post("http://localhost:11434/api/generate", json={
        "model": "llama3.2",
        "prompt": prompt,
        "stream": False,
        "temperature": 0.1
    })
    return resp.json()["response"]

print("Ready")

## 1. What is a Knowledge Graph?

Nodes = entities (concepts, people, documents)  
Edges = relationships ("is_a", "part_of", "related_to")

In [ ]:
# Build a simple knowledge graph about GenAI
G = nx.DiGraph()

# Add nodes
concepts = ["Transformer", "Self-Attention", "Multi-Head Attention",
            "Positional Encoding", "GPT", "Llama", "BERT",
            "RAG", "Vector DB", "ChromaDB", "Fine-tuning", "LoRA"]

for c in concepts:
    G.add_node(c, type="concept")

# Add edges
edges = [
    ("Transformer", "Self-Attention", "uses"),
    ("Transformer", "Multi-Head Attention", "contains"),
    ("Transformer", "Positional Encoding", "requires"),
    ("GPT", "Transformer", "based_on"),
    ("Llama", "Transformer", "based_on"),
    ("BERT", "Transformer", "based_on"),
    ("RAG", "Vector DB", "uses"),
    ("Vector DB", "ChromaDB", "example"),
    ("RAG", "GPT", "uses"),
    ("Fine-tuning", "LoRA", "technique"),
    ("Fine-tuning", "GPT", "modifies"),
]

for src, dst, rel in edges:
    G.add_edge(src, dst, relationship=rel)

print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

## 2. Traversing the Graph for Context

When asked about a concept, we can traverse the graph to find related concepts.

In [ ]:
def get_graph_context(entity, max_hops=2):
    """Get context by traversing the graph."""
    visited = set()
    context_parts = []
    
    def dfs(node, depth):
        if depth > max_hops or node in visited:
            return
        visited.add(node)
        
        for neighbor in G.neighbors(node):
            edge_data = G.get_edge_data(node, neighbor)
            if edge_data:
                context_parts.append(f"{node} {edge_data['relationship']} {neighbor}")
                dfs(neighbor, depth + 1)
        
        for pred in G.predecessors(node):
            if pred not in visited:
                edge_data = G.get_edge_data(pred, node)
                if edge_data:
                    context_parts.append(f"{pred} {edge_data['relationship']} {node}")
                dfs(pred, depth + 1)
    
    dfs(entity, 0)
    return "\n".join(context_parts)

context = get_graph_context("RAG")
print("Graph context for RAG:")
print(context)

## 3. Graph-Enhanced QA

Combine graph context with LLM for better answers.

In [ ]:
def graph_rag_qa(question, entity):
    context = get_graph_context(entity)
    prompt = f"""Answer the question using the knowledge graph context.

Knowledge Graph:
{context}

Question: {question}
Answer:"""
    return ollama_generate(prompt)

print(graph_rag_qa("What is RAG based on?", "RAG"))

## When to Use Graph RAG vs. Vector RAG

| Aspect | Vector RAG | Graph RAG |
|---|---|---|
| Best for | Semantic similarity, unstructured text | Relationships, structured connections |
| Query | "Find similar documents" | "Find connected entities" |
| Multi-hop | Weak (needs HyDE/multi-query) | Strong (graph traversal) |
| Construction | Easy (embed + store) | Hard (need to extract entities + relations) |

**Hybrid approach**: Vector RAG for retrieval + Graph RAG for relationship traversal.